# Exploratory Data Analysis (EDA)

This notebook contains exploratory analysis for the Xente transactions dataset. If `data/raw/data.csv` is not present, the cells will raise a clear error — replace or mount the dataset and re-run the notebook.

In [1]:
# Imports and data loading
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
possible_paths = [Path('data/raw/data.csv'), Path('../data/raw/data.csv'), Path('..\data\raw\data.csv')]
data_file = next((p for p in possible_paths if p.exists()), None)
if data_file is None:
    print('Warning: data/raw/data.csv not found. Cells below will be placeholders until dataset is available.')
    df = pd.DataFrame()
else:
    df = pd.read_csv(data_file, parse_dates=['TransactionStartTime'], low_memory=False)
    print('Loaded', data_file, 'with shape', df.shape)
df.head()

<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\hp\AppData\Local\Temp\ipykernel_4928\3342968043.py:7: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  possible_paths = [Path('data/raw/data.csv'), Path('../data/raw/data.csv'), Path('..\data\raw\data.csv')]


Loaded ..\data\raw\data.csv with shape (95662, 16)


,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15 02:18:49+00:00,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15 02:19:08+00:00,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15 02:44:21+00:00,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15 03:32:55+00:00,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15 03:34:21+00:00,2,0


In [ ]:
# Overview: rows, columns, dtypes
if df.empty:
    print('Dataframe is empty — load the CSV to run EDA')
else:
    display(df.info())
    display(df.describe(include='all'))
    display(df.nunique())

In [ ]:
# Missing values and basic cleaning hints
if df.empty:
    pass
else:
    missing = df.isnull().sum().sort_values(ascending=False)
    display(missing[missing>0])

In [ ]:
# Distribution of key numeric features
if df.empty:
    pass
else:
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    print('Numeric columns:', numeric)
    df[numeric].hist(figsize=(12, 8)); plt.tight_layout()

In [ ]:
# Categorical distributions (top categories)
if df.empty:
    pass
else:
    cats = df.select_dtypes(include=['object', 'category']).columns.tolist()
    for c in cats[:6]:
        print('---', c, '---')
        display(df[c].value_counts().head(10))

In [ ]:
# Correlation matrix for numeric features
if df.empty:
    pass
else:
    corr = df.corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Numeric feature correlations')

In [ ]:
# Outlier detection example for Transaction amount
if df.empty:
    pass
else:
    plt.figure(figsize=(8,3))
    sns.boxplot(x=df['Amount'].dropna())
    plt.title('Boxplot - Amount')

In [ ]:
# RFM feature skeleton: compute per-customer aggregates
if df.empty:
    rfm = pd.DataFrame()
else:
    snapshot_date = df['TransactionStartTime'].max() + pd.Timedelta(days=1)
    cust = df.groupby('CustomerId').agg({
        'TransactionStartTime':'max',
        'Amount':['sum','mean','std','count']
    })
    cust.columns = ['last_txn','total_amount','avg_amount','std_amount','tx_count']
    cust['recency_days'] = (snapshot_date - cust['last_txn']).dt.days
    rfm = cust.reset_index()
    display(rfm.head())

## Top 3–5 Insights

- Dataset has 95662 transactions for 3742 unique customers between 2018-11-15 and 2019-02-13.
- Amount distribution is highly skewed (median=1000, mean≈6717, max=9,880,000).
- Transactions per customer: median=7, but some customers have very high activity (max=4091).
- RFM medians — recency=25 days, frequency=7 tx, monetary=20000 (per-customer median total).
- Top product categories: financial_services and airtime dominate the transactions.